# Setup

In [1]:
from pathlib import Path
from dataclasses import dataclass, asdict
import random
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix
from torchvision import models
from torchvision.models import ConvNeXt_Tiny_Weights, EfficientNet_B3_Weights
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Config

In [ ]:
@dataclass
class CFG:
    seed: int = 42
    model_name: str = "convnext_tiny"  # convnext_tiny | efficientnet_b3
    img_size: int = 224
    batch_size: int = 16
    num_workers: int = 0
    num_workers_test: int = 0
    num_workers_val: int = 0
    epochs: int = 8
    lr: float = 3e-4
    weight_decay: float = 1e-4
    patience: int = 3
    n_splits: int = 5
    run_full_cv: bool = True
    mixed_precision: bool = True

cfg = CFG()
seed_everything(cfg.seed)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SPLIT_PATH = PROJECT_ROOT / "data" / "splits" / "train_5fold_stratified_group_seed42.csv"
TEST_META_PATH = PROJECT_ROOT / "data" / "processed" / "clean" / "test_metadata_with_hash.csv"
SAMPLE_SUB_PATH = PROJECT_ROOT / "data" / "raw" / "samplesubmission.csv"

CHECKPOINT_DIR = PROJECT_ROOT / "experiments" / "checkpoints" / "baseline"
OOF_DIR = PROJECT_ROOT / "experiments" / "oof_predictions"
SUB_DIR = PROJECT_ROOT / "output" / "submissions"
for p in [CHECKPOINT_DIR, OOF_DIR, SUB_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FOLDS_TO_RUN = list(range(cfg.n_splits)) if cfg.run_full_cv else [0]

print(f"DEVICE: {DEVICE}")
print(f"FOLDS_TO_RUN: {FOLDS_TO_RUN}")
print(f"Using split file: {SPLIT_PATH}")
print(f"num_workers(train/val/test): {cfg.num_workers}/{cfg.num_workers_val}/{cfg.num_workers_test}")

DEVICE: cuda
FOLDS_TO_RUN: [0, 1, 2, 3, 4]
Using split file: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\data\splits\train_5fold_stratified_group_seed42.csv
num_workers(train/val/test): 0/0/0


# Data

In [3]:
if not SPLIT_PATH.exists():
    raise FileNotFoundError("Split file tidak ditemukan. Jalankan notebook 02_data_split_prep.ipynb dulu.")
if not TEST_META_PATH.exists():
    raise FileNotFoundError("test_metadata_with_hash.csv tidak ditemukan. Jalankan notebook 02_data_split_prep.ipynb dulu.")

split_df = pd.read_csv(SPLIT_PATH).reset_index(drop=True)
split_df["row_id"] = np.arange(len(split_df))
test_df = pd.read_csv(TEST_META_PATH).reset_index(drop=True)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

labels = sorted(split_df["label"].unique().tolist())
label2idx = {label: i for i, label in enumerate(labels)}
idx2label = {i: label for label, i in label2idx.items()}
split_df["label_idx"] = split_df["label"].map(label2idx).astype(int)

print(f"Train rows in split: {len(split_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Num classes: {len(labels)}")
display(split_df.head())

Train rows in split: 1342
Test rows: 404
Num classes: 6


,label,file_name,file_path,ext,file_size,md5,fold,row_id,label_idx
0,fake_printed,printed_113.jpg,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,.jpg,503348,0009e30b22fee6b9032466fd61cc3658,2,0,2
1,fake_mannequin,mannequin_070.jpg,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,.jpg,84382,0010225da4adf0fc21f41b1274baad5c,4,1,0
2,realperson,real_143.jpg,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,.jpg,551953,004c025fa0e94f1bfeb4c105e844e591,3,2,5
3,fake_mannequin,mannequin_216.jpg,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,.jpg,89012,0074ac79a98d1ecd4897e5958b44c281,0,3,0
4,fake_unknown,unknown_206.jpeg,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,.jpeg,29190,0074bbd6358f3ae0985af0fb26651361,0,4,4


# Dataset And Model

In [4]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tfms = A.Compose([
    A.Resize(cfg.img_size, cfg.img_size),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.ColorJitter(p=0.3),
    A.ImageCompression(quality_range=(70, 100), p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

valid_tfms = A.Compose([
    A.Resize(cfg.img_size, cfg.img_size),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

class FaceDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, is_test: bool = False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["file_path"]).convert("RGB")
        img = np.array(img)
        img = self.transform(image=img)["image"]
        if self.is_test:
            return img, row["file_name"]
        return img, int(row["label_idx"]), int(row["row_id"])

def build_model(model_name: str, num_classes: int):
    model_name = model_name.lower()
    if model_name == "convnext_tiny":
        model = models.convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)
        return model
    if model_name == "efficientnet_b3":
        model = models.efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        return model
    raise ValueError(f"Unsupported model_name: {model_name}")

def make_loader(df, transform, is_test=False, shuffle=False):
    ds = FaceDataset(df=df, transform=transform, is_test=is_test)
    if is_test:
        workers = cfg.num_workers_test
    else:
        workers = cfg.num_workers if shuffle else cfg.num_workers_val

    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )

class_counts = split_df["label"].value_counts()
class_weight_values = []
for label in labels:
    c = class_counts[label]
    class_weight_values.append(len(split_df) / (len(labels) * c))
class_weights = torch.tensor(class_weight_values, dtype=torch.float32)
print("Class weights:", dict(zip(labels, [round(v, 4) for v in class_weight_values])))

Class weights: {'fake_mannequin': np.float64(1.2426), 'fake_mask': np.float64(0.9242), 'fake_printed': np.float64(2.9822), 'fake_screen': np.float64(1.1897), 'fake_unknown': np.float64(0.8223), 'realperson': np.float64(0.581)}


# Train Eval Utils

In [5]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, total=len(loader), desc="train", leave=False)
    for images, targets, _ in pbar:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=cfg.mixed_precision and DEVICE.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader.dataset)

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    probs_all = []
    y_all = []
    row_id_all = []

    pbar = tqdm(loader, total=len(loader), desc="valid", leave=False)
    for images, targets, row_ids in pbar:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        with autocast(enabled=cfg.mixed_precision and DEVICE.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, targets)

        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        probs_all.append(probs)
        y_all.append(targets.detach().cpu().numpy())
        row_id_all.append(row_ids.numpy())

        running_loss += loss.item() * images.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    probs_all = np.concatenate(probs_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    row_id_all = np.concatenate(row_id_all, axis=0)
    avg_loss = running_loss / len(loader.dataset)
    return avg_loss, y_all, probs_all, row_id_all

@torch.no_grad()
def predict_test(model, loader):
    model.eval()
    probs_all = []
    file_all = []

    pbar = tqdm(loader, total=len(loader), desc="test", leave=False)
    for images, file_names in pbar:
        images = images.to(DEVICE, non_blocking=True)
        with autocast(enabled=cfg.mixed_precision and DEVICE.type == "cuda"):
            logits = model(images)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        probs_all.append(probs)
        file_all.extend(list(file_names))

    probs_all = np.concatenate(probs_all, axis=0)
    return probs_all, file_all

# Dataloader Sanity Check

In [6]:
sanity_fold = FOLDS_TO_RUN[0]
sanity_trn = split_df[split_df["fold"] != sanity_fold].reset_index(drop=True)
sanity_val = split_df[split_df["fold"] == sanity_fold].reset_index(drop=True)

sanity_trn_loader = make_loader(sanity_trn, transform=train_tfms, is_test=False, shuffle=True)
sanity_val_loader = make_loader(sanity_val, transform=valid_tfms, is_test=False, shuffle=False)
sanity_test_loader = make_loader(test_df, transform=valid_tfms, is_test=True, shuffle=False)

t0 = time.time()
x, y, rid = next(iter(sanity_trn_loader))
t1 = time.time()
print(f"Train batch loaded in {t1 - t0:.2f}s | x={tuple(x.shape)} y={tuple(y.shape)} rid={tuple(rid.shape)}")

t0 = time.time()
xv, yv, ridv = next(iter(sanity_val_loader))
t1 = time.time()
print(f"Val batch loaded in {t1 - t0:.2f}s | x={tuple(xv.shape)} y={tuple(yv.shape)}")

t0 = time.time()
xt, fn = next(iter(sanity_test_loader))
t1 = time.time()
print(f"Test batch loaded in {t1 - t0:.2f}s | x={tuple(xt.shape)} files={len(fn)}")

Train batch loaded in 0.42s | x=(16, 3, 224, 224) y=(16,) rid=(16,)
Val batch loaded in 0.27s | x=(16, 3, 224, 224) y=(16,)
Test batch loaded in 0.25s | x=(16, 3, 224, 224) files=16


# Train

In [7]:
timestamp = time.strftime("%Y%m%d_%H%M%S")
run_name = f"baseline_{cfg.model_name}_img{cfg.img_size}_{timestamp}"
run_ckpt_dir = CHECKPOINT_DIR / run_name
run_ckpt_dir.mkdir(parents=True, exist_ok=True)

oof_probs = np.full((len(split_df), len(labels)), np.nan, dtype=np.float32)
oof_targets = split_df["label_idx"].values.copy()
test_probs_sum = np.zeros((len(test_df), len(labels)), dtype=np.float32)
fold_scores = []

test_loader = make_loader(test_df, transform=valid_tfms, is_test=True, shuffle=False)
scaler = GradScaler(enabled=cfg.mixed_precision and DEVICE.type == "cuda")

for fold in FOLDS_TO_RUN:
    print("=" * 80)
    print(f"Fold {fold}")

    trn_fold = split_df[split_df["fold"] != fold].reset_index(drop=True)
    val_fold = split_df[split_df["fold"] == fold].reset_index(drop=True)

    trn_loader = make_loader(trn_fold, transform=train_tfms, is_test=False, shuffle=True)
    val_loader = make_loader(val_fold, transform=valid_tfms, is_test=False, shuffle=False)

    model = build_model(cfg.model_name, num_classes=len(labels)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.lr * 0.1)

    best_f1 = -1.0
    best_epoch = -1
    best_path = run_ckpt_dir / f"fold{fold}_best.pt"
    patience_left = cfg.patience

    for epoch in range(1, cfg.epochs + 1):
        trn_loss = train_one_epoch(model, trn_loader, criterion, optimizer, scaler)
        val_loss, y_true, val_probs, row_ids = eval_one_epoch(model, val_loader, criterion)
        y_pred = val_probs.argmax(axis=1)
        val_f1 = f1_score(y_true, y_pred, average="macro")

        print(
            f"Epoch {epoch:02d} | trn_loss={trn_loss:.4f} | val_loss={val_loss:.4f} | val_macro_f1={val_f1:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch
            patience_left = cfg.patience
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "labels": labels,
                    "label2idx": label2idx,
                    "config": asdict(cfg),
                    "fold": fold,
                    "best_epoch": best_epoch,
                    "best_f1": best_f1,
                },
                best_path,
            )
        else:
            patience_left -= 1
            if patience_left <= 0:
                print("Early stopping triggered")
                break

        scheduler.step()

    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])

    _, y_true, val_probs, row_ids = eval_one_epoch(model, val_loader, criterion)
    y_pred = val_probs.argmax(axis=1)
    val_f1 = f1_score(y_true, y_pred, average="macro")
    fold_scores.append({"fold": fold, "best_epoch": best_epoch, "macro_f1": val_f1})

    oof_probs[row_ids] = val_probs

    test_probs, test_files = predict_test(model, test_loader)
    test_probs_sum += test_probs

    print(f"Fold {fold} done | best_epoch={best_epoch} | macro_f1={val_f1:.4f}")

test_probs_avg = test_probs_sum / max(len(FOLDS_TO_RUN), 1)
fold_scores_df = pd.DataFrame(fold_scores)
display(fold_scores_df)

Fold 0


Epoch 01 | trn_loss=1.0091 | val_loss=0.6145 | val_macro_f1=0.7654


Epoch 02 | trn_loss=0.5551 | val_loss=0.5425 | val_macro_f1=0.8156


Epoch 03 | trn_loss=0.3638 | val_loss=0.5899 | val_macro_f1=0.8193


Epoch 04 | trn_loss=0.2497 | val_loss=0.4910 | val_macro_f1=0.8705


...

Epoch 06 | trn_loss=0.1040 | val_loss=0.3636 | val_macro_f1=0.8881


Epoch 07 | trn_loss=0.0563 | val_loss=0.3758 | val_macro_f1=0.8770


Epoch 08 | trn_loss=0.0498 | val_loss=0.3268 | val_macro_f1=0.8877


Fold 4 done | best_epoch=6 | macro_f1=0.8881


,fold,best_epoch,macro_f1
0,0,6,0.904383
1,1,8,0.910243
2,2,7,0.887992
3,3,7,0.870923
4,4,6,0.888113


# Evaluate OOF

In [8]:
valid_mask = ~np.isnan(oof_probs).any(axis=1)
oof_eval_df = split_df.loc[valid_mask].copy()
oof_eval_probs = oof_probs[valid_mask]
oof_eval_true = oof_eval_df["label_idx"].values
oof_eval_pred = oof_eval_probs.argmax(axis=1)

oof_macro_f1 = f1_score(oof_eval_true, oof_eval_pred, average="macro")
pr, rc, f1c, sup = precision_recall_fscore_support(
    oof_eval_true,
    oof_eval_pred,
    labels=list(range(len(labels))),
    zero_division=0,
    average=None,
)
metrics_df = pd.DataFrame({
    "label": [idx2label[i] for i in range(len(labels))],
    "precision": pr,
    "recall": rc,
    "f1": f1c,
    "support": sup,
})

print(f"OOF evaluated rows: {len(oof_eval_df)} / {len(split_df)}")
print(f"OOF Macro F1: {oof_macro_f1:.5f}")
display(metrics_df)

cm = confusion_matrix(oof_eval_true, oof_eval_pred, labels=list(range(len(labels))))
cm_df = pd.DataFrame(cm, index=[idx2label[i] for i in range(len(labels))], columns=[idx2label[i] for i in range(len(labels))])
display(cm_df)

oof_prob_cols = [f"prob_{idx2label[i]}" for i in range(len(labels))]
oof_save_df = oof_eval_df[["row_id", "file_path", "label", "fold"]].copy()
oof_save_df["pred_label"] = [idx2label[i] for i in oof_eval_pred]
oof_save_df[oof_prob_cols] = oof_eval_probs
oof_save_path = OOF_DIR / f"oof_{run_name}.csv"
oof_save_df.to_csv(oof_save_path, index=False)
print(f"Saved OOF predictions: {oof_save_path}")

OOF evaluated rows: 1342 / 1342
OOF Macro F1: 0.89254


,label,precision,recall,f1,support
0,fake_mannequin,0.879121,0.888889,0.883978,180
1,fake_mask,0.861111,0.896694,0.878543,242
2,fake_printed,0.807692,0.840000,0.823529,75
3,fake_screen,0.955801,0.920213,0.937669,188
4,fake_unknown,0.920000,0.930147,0.925046,272
5,realperson,0.919786,0.893506,0.906456,385


,fake_mannequin,fake_mask,fake_printed,fake_screen,fake_unknown,realperson
fake_mannequin,160,8,1,0,7,4
fake_mask,5,217,3,2,4,11
fake_printed,3,1,63,1,3,4
fake_screen,3,0,1,173,3,8
fake_unknown,8,4,4,0,253,3
realperson,3,22,6,5,5,344


Saved OOF predictions: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\experiments\oof_predictions\oof_baseline_convnext_tiny_img224_20260328_011405.csv


# Submission

In [9]:
test_pred_idx = test_probs_avg.argmax(axis=1)
test_pred_label = [idx2label[i] for i in test_pred_idx]

test_pred_df = pd.DataFrame({
    "id": [Path(fn).stem for fn in test_files],
    "label": test_pred_label,
})

submission_df = sample_sub[["id"]].merge(test_pred_df, on="id", how="left")
if submission_df["label"].isna().any():
    missing_n = int(submission_df["label"].isna().sum())
    raise ValueError(f"Ada {missing_n} id test yang belum terprediksi")

sub_path = SUB_DIR / f"submission_{run_name}.csv"
submission_df.to_csv(sub_path, index=False)
print(f"Saved submission: {sub_path}")
display(submission_df.head())

config_path = run_ckpt_dir / "config_used.json"
pd.Series(asdict(cfg)).to_json(config_path, indent=2)
print(f"Saved config: {config_path}")

Saved submission: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\output\submissions\submission_baseline_convnext_tiny_img224_20260328_011405.csv


,id,label
0,test_001,fake_screen
1,test_002,fake_screen
2,test_003,realperson
3,test_004,realperson
4,test_005,fake_printed


Saved config: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\experiments\checkpoints\baseline\baseline_convnext_tiny_img224_20260328_011405\config_used.json
